# MAGIC Gamma Telescope Classification

The MAGIC telescope records showers of particles in the atmosphere. Gamma-ray showers (the signal astronomers want) must be separated from hadron showers (the background) using 10 image-shape measurements of each shower. With 19,020 events, this is the largest dataset in this batch.

In [1]:
import io, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def download(url):
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(req, timeout=180) as r:
        return r.read()
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

# UCI MAGIC Gamma Telescope (id 159)
df = pd.read_csv(io.BytesIO(download('https://archive.ics.uci.edu/static/public/159/data.csv')))
print(df.shape)
print('Events per class (g = gamma signal, h = hadron background):')
print(df['class'].value_counts())
df.head(3)


(19020, 11)
Events per class (g = gamma signal, h = hadron background):
class
g    12332
h     6688
Name: count, dtype: int64


,fLength,fWidth,fSize,fConc,fConc1,fAsym,fM3Long,fM3Trans,fAlpha,fDist,class
0,28.7967,16.0021,2.6449,0.3918,0.1982,27.7004,22.0110,-8.2027,40.0920,81.8828,g
1,31.6036,11.7235,2.5185,0.5303,0.3773,26.2722,23.8238,-9.9574,6.3609,205.2610,g
2,162.0520,136.0310,4.0612,0.0374,0.0187,116.7410,-64.8580,-45.2160,76.9600,256.7880,g


In [2]:

feats = ['fLength', 'fWidth', 'fSize', 'fConc', 'fConc1', 'fAsym', 'fM3Long', 'fM3Trans', 'fAlpha', 'fDist']
X = df[feats]; y = df['class'].map({'g': 'gamma', 'h': 'hadron'})
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
rf = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1).fit(Xtr, ytr)
lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(Xtr, ytr)
acc_rf = accuracy_score(yte, rf.predict(Xte)); acc_lr = accuracy_score(yte, lr.predict(Xte))
print(f'Random Forest accuracy: {acc_rf:.4f}')
print(f'Logistic Regression accuracy: {acc_lr:.4f}')
print(classification_report(yte, rf.predict(Xte)))


Random Forest accuracy: 0.8835
Logistic Regression accuracy: 0.7979


              precision    recall  f1-score   support

       gamma       0.89      0.94      0.91      3083
      hadron       0.88      0.78      0.82      1672

    accuracy                           0.88      4755
   macro avg       0.88      0.86      0.87      4755
weighted avg       0.88      0.88      0.88      4755



In [3]:
imp = pd.Series(rf.feature_importances_, index=feats).sort_values(ascending=False)
print(imp)
imp.plot(kind='barh', title='Which shower-shape measurements separate gamma from hadron?')
plt.tight_layout()
plt.show()
winner = rf if acc_rf >= acc_lr else lr
winner_name = 'Random Forest' if acc_rf >= acc_lr else 'Logistic Regression'
print(f'Winner: {winner_name}')
cm = confusion_matrix(yte, winner.predict(Xte), labels=sorted(y.unique()))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=sorted(y.unique()))
disp.plot(xticks_rotation=45)
plt.title(f'Confusion matrix - {winner_name}')
plt.tight_layout()
plt.savefig('confusion_matrix.png')
plt.show()


fAlpha      0.235244
fLength     0.159088
fWidth      0.120366
fSize       0.112136
fM3Long     0.085194
fDist       0.068777
fConc       0.061172
fAsym       0.055251
fConc1      0.053860
fM3Trans    0.048914
dtype: float64


Winner: Random Forest
